# CARMA-DACA — Bearing Fault Diagnosis (CWRU, missing data + changing conditions)

One-click Colab walkthrough for the paper **"A class alignment method based on graph convolution neural network for bearing fault diagnosis in presence of missing data and changing working conditions"** 
(*Measurement* 199 (2022) 111536, [DOI](https://doi.org/10.1016/j.measurement.2022.111536)).

This notebook installs the dependencies, gets the code and the CWRU data (12 kHz **and** 48 kHz), runs a quick **smoke test**, then trains the proposed **CARMA-DACA** method and the comparison methods, and reproduces the ablations / statistics / visualisations.

> **Tip:** set the runtime to GPU via *Runtime → Change runtime type → Hardware accelerator → GPU*.

## 1. Check the runtime (GPU)

In [ ]:
import torch, subprocess
print('PyTorch (if already installed):', getattr(torch, '__version__', 'not yet'))
try:
    print(subprocess.check_output(['nvidia-smi']).decode().split('\n')[0:10][8])
except Exception:
    print('nvidia-smi not available — you may be on CPU (training will be slow).')

## 2. Get the code

Run **Option A** (clone from GitHub) *or* **Option B** (upload the project zip), then continue.

In [ ]:
# Option A — clone from GitHub (replace <your-username> with the actual repo owner)
!git clone https://github.com/<your-username>/CARMA-DACA-Bearing-Fault-Diagnosis.git
%cd CARMA-DACA-Bearing-Fault-Diagnosis

In [ ]:
# Option B — upload CARMA-DACA-Bearing-Fault-Diagnosis.zip instead of cloning
# from google.colab import files
# up = files.upload()                       # choose the project .zip
# import zipfile, os
# z = next(iter(up))
# zipfile.ZipFile(z).extractall('.')
# %cd CARMA-DACA-Bearing-Fault-Diagnosis

## 3. Install dependencies

`torch` is pre-installed on Colab. We add the scientific stack and **PyTorch Geometric** (for `ARMAConv` / `ChebConv`).

In [ ]:
!pip install -q -r requirements.txt
!pip install -q torch_geometric
print('\nInstall finished.')

In [ ]:
# verify the environment and the repo modules import cleanly
import torch
from torch_geometric.nn import ARMAConv, ChebConv
import config, data, model, mklmmd, da_losses, methods, train
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
print('methods:', methods.ALL_METHODS)
print('tasks  : case1', data.CASE1_TASKS, '| case2 has', len(data.CASE2_TASKS), 'tasks')

## 4. Download the CWRU data (12 kHz + 48 kHz)

Fetches the exact `.mat` files for the source (12 kHz) and target (48 kHz) domains, plus the Normal baseline, into `./CWRU` and validates each.

In [ ]:
!python download_cwru.py --root ./CWRU

## 5. Smoke test (fast — just checks the full pipeline runs)

A tiny 3-epoch run of the proposed method on one task. This verifies data loading, the missing-data mask, the ARMA graph branch, and the three-loss training loop end-to-end before you commit to a full 500-epoch run.

In [ ]:
!python train.py --method CARMA-DACA --source C1 --target C2 --label 0.10 --epochs 3

## 6. Train the proposed method (full run)

Case study 1 (missing data only), task `C1 → C2`, 10 % labelled target, 500 epochs. 
Change `--source/--target/--label/--epochs` as you like, or use `--case case1` / `--case case2` to sweep a whole case study.

In [ ]:
!python train.py --method CARMA-DACA --source C1 --target C2 --label 0.10 --epochs 500

## 7. Full comparison benchmark (every method × every task)

Runs all nine methods across a case study and writes a CSV to `results/`. 
Use `--runs 10` to match the paper's 10-run averaging (slower).

In [ ]:
!python train.py --benchmark --case case2 --label 0.10 --runs 1
import pandas as pd, glob
csv = sorted(glob.glob('results/benchmark_*.csv'))[-1]
print(csv); pd.read_csv(csv)

## 8. Ablations, statistics, and visualisations

- **`ablation.py`** — `alpha`/`beta` grid (Fig. 13), A-/AL-distance (Fig. 12), label-fraction sweep.
- **`stats.py`** — Friedman + Nemenyi test and critical-difference diagram (Fig. 11).
- **`tsne.py` / `confusion.py`** — feature t-SNE (Fig. 6/10) and confusion matrix (Fig. 5/9).

In [ ]:
# label-fraction sweep for the proposed method (0/1/5/10%)
!python ablation.py labels --source A1 --target D2 --method CARMA-DACA --epochs 100 --runs 1

In [ ]:
# A-distance / AL-distance of the learned features
!python ablation.py adistance --source C1 --target D2 --method CARMA-DACA --epochs 100

In [ ]:
# statistical comparison (needs one or more benchmark CSVs from cell 7)
!python stats.py results/benchmark_case2_label10.csv
from IPython.display import Image
Image('results/cd_diagram.png')

In [ ]:
# t-SNE and confusion matrix for one task
!python tsne.py --source C1 --target C2 --label 0.10 --epochs 100
!python confusion.py --source D1 --target D2 --label 0.10 --epochs 100
from IPython.display import Image, display
display(Image('results/tsne_C1_C2.png'))
display(Image('results/confusion_D1_D2.png'))

---
### Notes
- Results are computed from **real training**; exact paper numbers depend on seed, library versions, and hardware (the paper averages each task over 10 runs).
- All ambiguous design points are exposed in `config.py` — see the README's *"A note on faithful reproduction"* section.

**Citation:** M. Kavianpour, A. Ramezani, M. T. H. Beheshti, *Measurement* 199 (2022) 111536. 
[doi:10.1016/j.measurement.2022.111536](https://doi.org/10.1016/j.measurement.2022.111536)